# 🔹 Notebook 5 — Data Governance con Knowledge Graph

**Obiettivo**: Implementare query di governance — data lineage, PII detection, audit trail — usando il KG.

Questo notebook corrisponde alla sezione **"Compliance e Data Governance"** del libro.


## Setup e seed dati di governance

In [ ]:
import os
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=('neo4j', os.getenv('NEO4J_PASSWORD', 'yourpassword'))
)

def run(q, **params):
    with driver.session() as s:
        return [r.data() for r in s.run(q, params)]

# Seed: modello di governance
seed = [
    "MERGE (crm:DataSource {name:'CRM Salesforce', type:'SaaS'})",
    "MERGE (erp:DataSource {name:'SAP ERP', type:'OnPrem'})",
    "MERGE (hr:DataSource {name:'HR System', type:'SaaS'})",
    "MERGE (etl:ETLJob {name:'ETL_CRM_to_DW', schedule:'daily'})",
    "MERGE (dw:Dataset {name:'Customer_360', classification:'Confidential'})",
    "MERGE (f1:DataField {name:'customer_email', classification:'PII', gdpr_article:'Art.6'})",
    "MERGE (f2:DataField {name:'purchase_history', classification:'Internal'})",
    "MERGE (f3:DataField {name:'employee_salary', classification:'PII', gdpr_article:'Art.9'})",
    "MERGE (app1:Application {name:'Marketing Dashboard'})",
    "MERGE (app2:Application {name:'HR Analytics'})",
    "MERGE (owner1:Person {name:'Maria Rossi', role:'Data Owner - Marketing'})",
    "MERGE (owner2:Person {name:'Luigi Bianchi', role:'DPO'})",
    "MERGE (policy:RetentionPolicy {name:'GDPR_Standard', duration:'36 mesi'})",
    "MERGE (basis:LegalBasis {type:'Consenso esplicito', gdpr_ref:'Art.6(1)(a)'})",
    # Relazioni lineage
    "MATCH (crm:DataSource {name:'CRM Salesforce'}),(etl:ETLJob {name:'ETL_CRM_to_DW'}) MERGE (crm)-[:FEEDS]->(etl)",
    "MATCH (etl:ETLJob {name:'ETL_CRM_to_DW'}),(dw:Dataset {name:'Customer_360'}) MERGE (etl)-[:PRODUCES]->(dw)",
    "MATCH (dw:Dataset {name:'Customer_360'}),(f1:DataField {name:'customer_email'}) MERGE (dw)-[:CONTAINS]->(f1)",
    "MATCH (dw:Dataset {name:'Customer_360'}),(f2:DataField {name:'purchase_history'}) MERGE (dw)-[:CONTAINS]->(f2)",
    "MATCH (hr:DataSource {name:'HR System'}),(f3:DataField {name:'employee_salary'}) MERGE (hr)-[:CONTAINS]->(f3)",
    "MATCH (f1:DataField {name:'customer_email'}),(app1:Application {name:'Marketing Dashboard'}) MERGE (f1)-[:CONSUMED_BY]->(app1)",
    "MATCH (f3:DataField {name:'employee_salary'}),(app2:Application {name:'HR Analytics'}) MERGE (f3)-[:CONSUMED_BY]->(app2)",
    "MATCH (dw:Dataset {name:'Customer_360'}),(o:Person {name:'Maria Rossi'}) MERGE (dw)-[:OWNED_BY]->(o)",
    "MATCH (f3:DataField {name:'employee_salary'}),(o:Person {name:'Luigi Bianchi'}) MERGE (f3)-[:OVERSEEN_BY]->(o)",
    "MATCH (f1:DataField {name:'customer_email'}),(p:RetentionPolicy {name:'GDPR_Standard'}) MERGE (f1)-[:RETENTION]->(p)",
    "MATCH (f1:DataField {name:'customer_email'}),(b:LegalBasis {type:'Consenso esplicito'}) MERGE (f1)-[:LEGAL_BASIS]->(b)",
]

for q in seed:
    run(q)
print('Modello di governance caricato.')

## 5.1 — Data Lineage: traccia la provenienza dei dati

In [ ]:
# Query: da dove arrivano i dati PII nel Marketing Dashboard?
results = run("""
MATCH lineage = (source:DataSource)
  -[:FEEDS]->(etl:ETLJob)
  -[:PRODUCES]->(dataset:Dataset)
  -[:CONTAINS]->(field:DataField {classification: 'PII'})
  -[:CONSUMED_BY]->(app:Application)
RETURN source.name AS sorgente,
       etl.name AS pipeline,
       dataset.name AS dataset,
       field.name AS campo_pii,
       app.name AS applicazione
""")

print('Data Lineage — Campi PII:')
for r in results:
    print(f"  {r['sorgente']} -> {r['pipeline']} -> {r['dataset']} -> {r['campo_pii']} -> {r['applicazione']}")

## 5.2 — PII Detection: dove sono i dati personali?

In [ ]:
results = run("""
MATCH (f:DataField {classification: 'PII'})
OPTIONAL MATCH (f)<-[:CONTAINS]-(ds)
OPTIONAL MATCH (f)-[:CONSUMED_BY]->(app)
OPTIONAL MATCH (f)-[:OVERSEEN_BY]->(dpo)
RETURN f.name AS campo,
       f.gdpr_article AS articolo_gdpr,
       collect(DISTINCT ds.name) AS sorgenti,
       collect(DISTINCT app.name) AS applicazioni,
       collect(DISTINCT dpo.name) AS responsabili
""")

print('Mappa PII completa:')
for r in results:
    print(f"  Campo: {r['campo']} (GDPR {r['articolo_gdpr']})")
    print(f"    Sorgenti: {r['sorgenti']}")
    print(f"    Usato da: {r['applicazioni']}")
    print(f"    Responsabile: {r['responsabili']}")
    print()

## 5.3 — Report GDPR Art. 30: Registro dei Trattamenti

In [ ]:
results = run("""
MATCH (f:DataField {classification: 'PII'})
OPTIONAL MATCH (f)-[:LEGAL_BASIS]->(basis:LegalBasis)
OPTIONAL MATCH (f)-[:RETENTION]->(policy:RetentionPolicy)
OPTIONAL MATCH (f)<-[:CONTAINS]-(ds)-[:OWNED_BY]->(owner)
RETURN f.name AS dato_personale,
       f.gdpr_article AS base_normativa,
       basis.type AS base_giuridica,
       policy.duration AS conservazione,
       owner.name AS responsabile
""")

print('=== REGISTRO DEI TRATTAMENTI (GDPR Art. 30) ===')
print(f'{"Dato":25s} {"Base giuridica":25s} {"Conservazione":15s} {"Responsabile"}')
print('-' * 85)
for r in results:
    print(f"{r['dato_personale'] or '-':25s} {(r['base_giuridica'] or '-'):25s} {(r['conservazione'] or '-'):15s} {r['responsabile'] or '-'}")

## Cleanup

In [ ]:
driver.close()
print('Notebook completato.')